# 02 — Frame Labeling

**Primary path (free):** Load the aggregated CSV produced by `aggregate_frames.sql` and inspect the frame distributions. No per-article data needed.

**Secondary path (spot-checking only):** If you ran the full raw extraction (`extract_genai_gov.sql`), load those rows and run the Python preprocessing pipeline to manually verify that the frame labels make sense.

Start with Part A. Only run Part B if you want to audit individual article examples.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import load_agg, monthly_volume_agg, frame_shares_agg
from src.dictionaries import MILESTONES

sns.set_theme(style='whitegrid', palette='colorblind')

---
## Part A — Aggregated path (primary)

Run `aggregate_frames.sql` in the BigQuery console, download the result as CSV, and save to `data/processed/monthly_frames.csv`.

In [ ]:
AGG_PATH = Path('../data/processed/monthly_frames.csv')
agg_df = load_agg(AGG_PATH)
print(f'Loaded {len(agg_df)} months, {agg_df["total_articles"].sum():,} total articles')
agg_df.head()

In [ ]:
# Monthly totals
vol = monthly_volume_agg(agg_df)
print('Date range:', vol.index[0], '→', vol.index[-1])
print('Peak month:', vol.idxmax(), '—', vol.max(), 'articles')
vol.tail()

In [ ]:
# Frame hit totals across all months
frame_cols = [c for c in agg_df.columns if c.startswith('frame_')]
frame_totals = agg_df[frame_cols].sum().rename(index=lambda x: x.replace('frame_', ''))

fig, ax = plt.subplots(figsize=(8, 4))
frame_totals.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Total frame keyword matches across all months')
ax.set_xlabel('Number of matching articles')
plt.tight_layout()
plt.show()

print('\nFrame totals:')
print(frame_totals.sort_values(ascending=False))

In [ ]:
# Frame shares over time
shares = frame_shares_agg(agg_df)
shares.head()

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(12, 4))
dates = shares.index.to_timestamp()
ax.stackplot(
    dates,
    [shares[col] for col in shares.columns],
    labels=[c.replace('_', ' ').title() for c in shares.columns],
    alpha=0.85,
)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.set_ylabel('Frame share')
ax.set_ylim(0, 1)
ax.set_title('Frame distribution preview (before finalising for paper)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### Sanity checks

Look for:
- Any months with suspiciously high or low totals (data gaps)
- Frame shares that seem implausible (e.g. one frame dominating at >80%)
- A visible uptick around Nov 2022 (ChatGPT launch) — if absent, revisit the GenAI filter

In [ ]:
# Months with unusually low article counts (potential data gaps)
mean_vol = vol.mean()
low = vol[vol < mean_vol * 0.3]
if low.empty:
    print('No low-volume months detected.')
else:
    print('Potential data gaps:')
    print(low)

---
## Part B — Raw article spot-check (optional)

Only run this if you downloaded the full raw corpus via `extract_genai_gov.sql`.
Purpose: verify that individual articles labelled under each frame actually look right.

In [ ]:
RAW_PATH = Path('../data/raw/gdelt_genai_gov.csv')
if not RAW_PATH.exists():
    print('Raw corpus not found — skip this section.')
else:
    from src.preprocessing import load_raw, run_pipeline
    from src.dictionaries import FRAME_DICTS

    raw_df = load_raw(RAW_PATH)
    df = run_pipeline(raw_df)
    print(f'Loaded {len(df):,} articles after dedup')

In [ ]:
# Sample 3 articles per dominant frame to verify labels
if 'df' in dir() and 'dominant_frame' in df.columns:
    SPOT_COLS = ['DATE', 'SourceCommonName', 'dominant_frame', 'Quotations']
    for frame_name in FRAME_DICTS:
        n_frame = (df['dominant_frame'] == frame_name).sum()
        if n_frame == 0:
            continue
        sample = df[df['dominant_frame'] == frame_name][SPOT_COLS].sample(
            min(3, n_frame), random_state=42
        )
        print(f'\n=== {frame_name} (n={n_frame:,}) ===')
        for _, row in sample.iterrows():
            print(f'  [{row["DATE"]}] {row["SourceCommonName"]}')
            print(f'  {str(row["Quotations"])[:250]}')